# Part 1 - Data Warehouse Modeling: Star Schema

## 1.1 Define the Business Process and the Fact Grain

The Business process: Financial transactions of a trading activity recorded in the 2024 account statement.

The Grain of the Fact table: one row in "Fact_Transactions" represents one transaction from the account statement file. We can consider this an event transaction fact, one fact row per business event.


## 1.2 Identify Fact and Dimensions

We have identified the following tables:

- **Fact Table**: Fact_Transactions
  
- **Dimension Tables**:
  - Dim_Time
  - Dim_Geography
  - Dim_Symbol
  - Dim_TransactionType



In **Fact_Transactions** I put **Unit***, which is the measure of the fact, and **IDTransaction**, ia a descriptive attribute of the single transaction.
The other attributes of the account statemnet decsribe the fact becoming dimensions, **Date** tells when the transaction happended and goes into **Dim_Time**; **TransactionType** tells the type of events like Buy or Sell and goes into **Dim_TransactionType**;
**Symbol** tell about what the traded and goes into **Dim_Symbol**, togheter with ***company_name, industry and sector*** that describe the traded stock, comes from the symbol file.

The attribute **country** is the difficult one, becouse come from the 'symbol file' and in a first moment seems as an attribute of **Dim_Symbol**, but the assignment asks for a separate **Dim_Geography** and tells us to avoid unnecessary duplication trough dimensions, to this specific reason I decided to keep **country** only in **Dim_Geography**, togheter with ***region and sub-region*** from the 'country' file.

The symbols file is used only during the loading phase, to connect every transaction to the country of its company. Also I did not include in the model the columns of the country file that are not useful for any analysis


Attribute:

- Fact_Transaction: --> IDTransaction, Unit
- Dim_time: --> Date
- Dim_TransactionType: --> TransactionType
- Dim_Symbol: --> Symbol, company_name, sector, industry
- Dim_Geography: --> country, region, sub-region

## 1.3 Dimension Hierarchies

- Dim_Time: Day -> Month -> Quarter -> Year
- Dim_Geography: Country -> Region -> Sub-region
- Dim_Symbol: Symbol -> Industry -> Sector
- Dim_TransactionType: No hierarchy 


In Dim_Symbol, each simboly come from one industry, and each industry has just one sector, the the hierarchies is symbol, industry and sector

in Dim_TransactionType, we have no levels just an attribute with 2 value, so no hierarchy

## 1.4 Final Star Schema

**Fact_Transactions (time_sk, symbol_sk, geo_sk, type_sk, id_transaction, units)** contains 4 foreign key that aims inthe same dimensions

**Dim_Time (time_sk, date, day, month, quarter, year)** time_sk is the primary key and date is teh bottom

**Dim_Symbol (symbol_sk, symbol, company_name, industry, sector)**

**Dim_Geography (geo_sk, country, region, sub_region)**

**Dim_TransactionType (type_sk, transaction_type)** #just one attribute



Each dimension tablse has a surrogate primary key
- time_sk
- symbol_sk
- geo_sk
- type_sk

The fact table references all the dimensions tables thourght the foreign keys, instead the primary key is the composisione of the foreign key.
**units** is the measure of the fact aggregade with the SUM operation;
**id_transaction** is a descriptive attribute.
The number of of transactions is compued by couting the fact rows

# Part 2 - Data Transformation and Analysis


In [1]:
import pandas as pd

# to display more columns when we inspect the dataset
pd.set_option("display.max_columns", 100)

tx_raw = pd.read_csv("/Users/chris/Desktop/ChristianBDA2026/Homework/Homework3/Datasets/account-statement-1-1-2024-12-31-2024.csv", sep=';', encoding = 'utf-8-sig')


symbols = pd.read_csv('/Users/chris/Desktop/ChristianBDA2026/Homework/Homework3/Datasets/symbols.csv', sep=';', encoding = 'utf-8-sig')

country = pd.read_csv('/Users/chris/Desktop/ChristianBDA2026/Homework/Homework3/Datasets/country.csv')

print("Account statement shape:", tx_raw.shape)
print("Symbols shape:", symbols.shape)
print("Country shape:", country.shape)

tx_raw.head()

Account statement shape: (2745, 6)
Symbols shape: (3194, 5)
Country shape: (249, 11)


,IDTransaction,Date,TransactionType,Symbol,Unit,Unnamed: 5
0,2.769834e+09,11/01/2024 10:44:03,BUY,BAP,1605.0,NaN
1,2.767325e+09,24/01/2024 08:07:24,SELL,BAP,1605.0,NaN
2,2.815474e+09,10/01/2024 11:00:08,SELL,BAP,914.0,NaN
3,2.622244e+09,16/01/2024 08:14:21,BUY,ACGL,646.0,NaN
4,2.629871e+09,16/01/2024 14:34:12,SELL,ALVO,646.0,NaN


In [2]:
# check ispect columns and missing values

summary = pd.DataFrame({
    "column": tx_raw.columns,
    "dtype": tx_raw.dtypes.astype(str).values,
    "missing_values": tx_raw.isna().sum().values,
    "missing_pct": (tx_raw.isna().mean().values * 100).round(2)
})

summary

,column,dtype,missing_values,missing_pct
0,IDTransaction,float64,464,16.9
1,Date,str,464,16.9
2,TransactionType,str,464,16.9
3,Symbol,str,464,16.9
4,Unit,float64,464,16.9
5,Unnamed: 5,float64,2745,100.0


We can noticed an empty extra column every line of the file ends with ";" and a group of completely blank rows. 

In [8]:
# 4. Clean the transaction log


# We start from the raw dataset
tx = tx_raw.copy()

# Remove the empty extra column created by ";" and blanck rows
tx = tx.drop(columns=[c for c in tx.columns if c.startswith("Unnamed")])


tx = tx.dropna(how="all")

# Convert the columns to the correct types, so Dates are in day/month/year format

tx["IDTransaction"] = tx["IDTransaction"].astype("int64")
tx["Unit"] = tx["Unit"].astype("int64")
tx["Date"] = pd.to_datetime(tx["Date"], format="%d/%m/%Y %H:%M:%S")

print("Valid transactions:", len(tx))
print("Transaction types:", tx["TransactionType"].value_counts().to_dict())

Valid transactions: 2281
Transaction types: {'SELL': 1089, 'BUY': 1082, 'DIVIDENT': 110}


In [9]:


# 5. Verify that every transaction symbol exists in the symbol dataset


# transactions whose symbol is not in symbols.csv cannot be described
# by Dim_Symbol and Dim_Geography, so they must be identified
unknown = ~tx["Symbol"].isin(symbols["symbol"])

print("Transactions with unknown symbol:", unknown.sum(), "of", len(tx))
print("Symbols not found:", sorted(tx.loc[unknown, "Symbol"].unique()))

# We remove them, because they cannot be linked to the dimensions
tx = tx[~unknown].copy()
print("Transactions kept:", len(tx))

Transactions with unknown symbol: 212 of 2281
Symbols not found: ['AGO.l', 'ARCH', 'AZM', 'CCAP', 'CSIQ', 'FNC', 'HTGC', 'IBE', 'MFG', 'MONC', 'OBDC', 'RCMT', 'RIGZU', 'SAP', 'TKC', 'UCG', 'VWS', 'WF']
Transactions kept: 2069


In [ ]:
# 6. Verify that every company country can be mapped to the country dataset


# Two country names do not match the NAME_FIXES used in country.csv,
# so we fix them before the merge
NAME_FIXES = {"Turkey": "Türkiye", "Taiwan": "Taiwan, Province of China"}
symbols["country"] = symbols["country"].replace(NAME_FIXES)

# After the fix, every company country must be mappable.
not_mapped = set(symbols["country"]) - set(country["name"])
print("Countries not mappable:", not_mapped or "none")

Countries not mappable: none


## 2.2 Analytical Questions

